In [4]:
import ollama
from ollama import chat
import sqlite3

In [1]:
# Building the Tool_call Function
def query_product_database(sql_query: str) -> str:
    """Execute a SQL query on the product.db database and return the results.
    
    Args:
        sql_query: A valid SQLite SELECT query against the table 'product'.
        
    Returns:
        The rows returned from the database query as a string.
    """
    if not sql_query.strip().lower().startswith('select'):
        return "Error: only SELECT queries are allowed."
    
    try:
        # Connecting to the database
        conn = sqlite3.connect('product.db')
        cursor = conn.cursor()
        
        # Executing the query requested by the LLM
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        
        conn.close()
        return str(rows)
    except Exception as e:
        return f"Error executing query: {str(e)}"

In [6]:
# Initialize the chat messages with a question about your database
SYSTEM_PROMPT = """You answer questions about a product database using the query_product_database tool.
Always query the database before answering. Never guess.

Table: product
Columns:
- Product_Category (text): Kitchen Product, Home Appliance, Industrial Tool, Mechanical Accessory, Office Equipment, Smart Appliance, Wearable Device, Electronic Device
- Product_Subcategory (text): e.g. Juicer, Printer, Cutter, Gear Housing, Smart Watch
- Image_Width, Image_Height (int)
- Lighting_Level, Background_Complexity, Image_Noise_Level, Image_Brightness, Image_Contrast, Edge_Density (float)
- Texture_Complexity, Shape_Complexity, Contour_Length, Boundary_Smoothness, Surface_Curvature (float)
- Symmetry_Index, Aspect_Ratio, Compactness, Convexity (float)
- Region_Count (int), Largest_Region_Area (float), Foreground_Ratio (float)
- Boundary_Precision, Boundary_Recall, Mask_Consistency (float)
- Shape_Descriptor_Score, Surface_Feature_Score, Visual_Balance (float)
- Aesthetic_Score (0-10), Design_Innovation_Index, Manufacturing_Feasibility (float)
- CAD_Modification_Effort (float), Design_Optimization_Score (float)
- Inference_Time_ms, Memory_Consumption_MB (float)
- IoU, Dice_Coefficient, Segmentation_Accuracy (float)
- User_Preference (text): Low, Medium, High
- Product_Form_Class (text): Poor, Moderate, Good, Excellent

There is NO sales or quantity column. If asked about sales, say the data has none.
Write SQLite SELECT queries only. Answer in 9-10 lines maximum."""

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'Give me the details of the dataset'}
]

In [7]:
# Running the actual model
while True:
    response = chat(
        model='qwen3:8b',
        messages=messages,
        tools=[query_product_database],
        think=False,
        options={'num_predict':400}
    )
    
    # Append the assistant's response to history
    messages.append(response.message)
    
    # Print out what the LLM is thinking or saying
    if response.message.thinking:
        print("Thinking:", response.message.thinking)
    if response.message.content:
        print("Answer:", response.message.content)
        
    # Check if the LLM wants to use our database function
    if not response.message.tool_calls:
        break # Exit the loop if it has its final answer
        
    # If the LLM requested a tool call, execute it
    for call in response.message.tool_calls:
        if call.function.name == 'query_product_database':
            # Extract the SQL query the AI decided to write
            print(f"\n[AI generated SQL]: {call.function.arguments['sql_query']}")
            
            # Run your Python function with that SQL query
            result = query_product_database(**call.function.arguments)
            print(f"[Database Result]: {result}\n")
        else:
            result = 'Unknown tool'
            
        # Feed the database results back into the chat history
        messages.append({
            'role': 'tool', 
            'tool_name': call.function.name, 
            'content': result
        })

Answer: The dataset consists of product details with various attributes such as categories, subcategories, image characteristics, design scores, and performance metrics. Here are the details of the dataset:



[AI generated SQL]: SELECT * FROM product LIMIT 5;
[Database Result]: [('Kitchen Product', 'Juicer', 640, 1024, 0.515, 0.712, 0.028, 0.857, 0.951, 0.863, 0.592, 0.293, 1327.459, 0.762, 0.411, 0.288, 2.153, 0.806, 0.913, 23, 8927.734, 0.509, 0.962, 0.918, 0.792, 0.434, 0.721, 0.352, 4.633, 0.618, 0.675, 9.22, 51.826, 35.98, 186.061, 0.86, 0.885, 0.957, 'High', 'Moderate'), ('Home Appliance', 'Coffee Machine', 800, 640, 0.586, 0.702, 0.07, 0.387, 0.738, 0.266, 0.253, 0.395, 1063.818, 0.648, 0.789, 0.782, 1.67, 0.936, 0.777, 2, 7468.561, 0.547, 0.651, 0.874, 0.999, 0.361, 0.897, 0.828, 4.717, 0.462, 0.323, 2.242, 92.831, 106.365, 267.696, 0.859, 0.877, 0.82, 'High', 'Good'), ('Industrial Tool', 'Cutter', 640, 768, 0.928, 0.868, 0.04, 0.867, 0.876, 0.307, 0.332, 0.51, 1348.661, 0.778

KeyboardInterrupt: 